# Forecast verification: does what AirGyaan serves make sense?

This notebook runs the app's forecasting code end to end and checks its quality. It uses the **same functions the app runs**, not copies of them. It replaces `notebooks/archive/forecast_feasibility.ipynb`, which the Nov 2026 audit found had drifted from the app (audit §3: different data, filters, alpha, model choice and averaging).

**The two questions that matter most**

1. **Does the forecast beat the seasonal average (climatology)?** Climatology needs no recent reading at all, so a forecast that cannot beat it is not worth showing.
2. **How does that advantage decay as the input reading gets older?** OpenAQ publishes days late, so the app usually forecasts from a reading that is 2–5 days old.

Throughout, **skill** means *% lower error than the seasonal average*: `skill = 100 × (climatology error − forecast error) / climatology error`. Skill 0% means "no better than the seasonal average". The app labels a day **forecast** at ≥ 20% skill, **outlook** at ≥ 5%, and otherwise serves the seasonal average labelled **seasonal_normal**. Since Nov 2026 that skill is the **median over every eligible held-out year** (each trained only on the years before it), not one year.

**How to read the code.** Every line that calls app code carries a marker like `# ▶ APP CODE: scripts/forecast/nightly_forecast.py · build_station_rows`. Lines without a marker are only fetching, looping or plotting. The plumbing lives in `scripts/forecast/verification_helpers.py` (no forecasting logic in it). The notebook is **read-only**: its Supabase connection refuses every write.

### What each section checks, and the app code it calls

| § | Section | App code called |
|---|---|---|
| 1 | Load history the way the monthly refit does; daily city means | `baselines.load_history_from_db` (which applies `daily_quality.complete_days` and `daily_quality.station_day_aqi`) |
| 2 | Blend vs persistence vs climatology on every eligible held-out year, scored per station, and the median across years; the Pune check | `baselines.station_skill_years` (what `refit_params.fit_city` calls), `baselines.model_skill_table`, `baselines.fit_alpha`, `baselines.city_series` / `aqi_series` |
| 3 | Skill vs data age and horizon (heatmaps): the table that decides what is served | `baselines.station_skill_years(...)["modes"]` (what `refit_params.fit_city` stores in `forecast_modes`), `nightly_forecast.load_modes` |
| 4 | Forecast vs actual on a held-out season, using the serving function **and the bias correction**; band coverage (Check D); served fit vs scored fit (Check E) | `refit_params.fit_city` + `fit_stations`, `nightly_forecast.build_station_rows`, `bias_correction.city_day_errors` / `decide_all` / `apply_corrections`, `serving_mirror.expand_day` (bands), `baselines.model_skill_table`, `aqi_utils.compute_subindex` / `get_aqi_label` |
| 5 | Bias correction: what it changed and whether it helped (it learns only from *forecast*/*outlook* rows) | `nightly_forecast.load_scored_history`, `bias_correction.city_day_errors`, `bias_correction.decide_all`, `bias_correction.BIAS_CORRECT_MODES` |
| 6 | Gain from the diurnal (hourly) curve | `baselines.load_city_hourly` / `load_city_hourly_aqi`, `baselines.backtest_hour`, `baselines.complete_days` + `diurnal_shape` (as `fit_city` uses them) |
| 7 | Averaging over the 3 nearest stations (also what the Ask AI tools use) | `serving_mirror.nearest_stations` / `expand_day` / `average_forecast_days` (Python mirror of the TypeScript), `build_station_rows` (via §4) |
| 8 | Live match: recompute last night's forecast and compare with what was stored | `nightly_forecast.load_params`, `load_modes`, `fetch_daily_rows`, `simulate.latest_observation`, `build_station_rows` |
| 9 | Recent track record of served forecasts | `nightly_forecast.load_scored_history`, `load_params` |

### Limitations (each is repeated where it applies)

- **Sample locations, not every location.** The app answers for any point a user picks. §7 uses 3 randomly chosen points per city (fixed random seed), not every possible point.
- **The TypeScript serving layer is mirrored in Python.** `expandDay` and `averageForecastDays` run in TypeScript in the app. The notebook uses `scripts/forecast/serving_mirror.py`, a line-for-line copy. A shared test pins the copy to the TypeScript: `tests/fixtures/serving_mirror/cases.json` is checked by both `tests/test_serving_mirror.py` and `tests/api/serving-mirror.test.ts`. The nearest-station search uses a sphere where the database uses the ellipsoid (under 0.5% different).
- **Held-out seasons use parameters fitted before that season.** The live app fits on all history. Scoring a past season with parameters that had already seen it would be cheating, so §4 and §7 refit with the refit's own functions on the years before `REPLAY_YEAR`. §4 Check E measures how far that fit is from the all-history fit the app serves.
- **α in the skill test is not quite the served α.** Each held-out year's blend skill uses α fitted on the years before it; the app serves α fitted on all history. Fitting on the test year would leak it into the score, so this is deliberate. §2 prints the two side by side.
- **Hourly checks use city-mean hourly data** (XKDR, up to Sep 2025). The app applies the city's shape to every station. There is no local station-level hourly history to test that on.
- **§8 and §9 depend on what the nightly job has written.** Until the post-audit refit and nightly have run on `main`, the stored forecasts come from the old code. §8 detects this and says so instead of reporting a failure.

## 0. Settings

`QUICK = True` runs one small city (Pune), PM2.5 only, in about a minute once history is cached. Set `QUICK = False` for the full check: all seven cities, PM2.5 and AQI. The held-out years are no longer chosen here: §2 and §3 test **every year the refit's own rule allows** (`baselines.eligible_test_years`), exactly as the refit does. The full run should take roughly 8–15 minutes (the bias-correction replay, band check and the all-history refit in §4 added a few minutes in Nov 2026) (the multi-year skill test alone is about 1.5 minutes on the Nov 2026 history). The first history load of the day adds about 1–2 minutes.

In [ ]:
QUICK = False            # True: Pune, PM2.5, 3 held-out years.  False: everything.

ALL_CITIES = ["Delhi", "Mumbai", "Bengaluru", "Hyderabad", "Chennai", "Kolkata", "Pune"]

if QUICK:
    CITIES = ["Pune"]
    POLLUTANTS = ["pm25"]
else:
    CITIES = ALL_CITIES
    POLLUTANTS = ["pm25", "aqi"]

# The one season §4, §6 and §7 replay through the serving function. Only a
# choice for these replays: the app's skill and labels use every eligible
# year (§2, §3). 2024 is eligible for every city.
REPLAY_YEAR = 2024

# The age of the input reading used in the forecast-vs-actual plots (§4, §7).
# Amrita's check (Nov 2026): the MEDIAN served data age is 1 day. In the first
# half of Nov 2026 the stored forecasts started from a 1-day-old reading almost
# everywhere (§9 prints this). Those were made by the old code,
# which did not skip half-published days, so with the completeness rule the age
# may grow. §3 shows every age from 0 to 7.
DATA_AGE_FOR_PLOTS = 1
AGES_BY_YEAR = (1, 3)           # §2: the two data ages shown year by year

LOCATIONS_PER_CITY = 3          # §7: random points per city
SAMPLE_STATIONS_PER_CITY = 5    # §8: stations whose live forecast is recomputed
RECENT_NIGHTS = 14              # §9: how many recent nights to score
RANDOM_SEED = 42                # fixed, so the random picks are the same every run
REFRESH_HISTORY = False         # True re-reads history even if today's cache exists

In [ ]:
# ── Setup: make the repo importable and load the app's modules ──────────────
import sys, time, warnings
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

from scripts.forecast import baselines as B              # skill tables, climatology, loaders
from scripts.forecast import model as M                  # the forecast formula and its constants
from scripts.forecast import nightly_forecast as nf      # the serving function
from scripts.forecast import refit_params as rp          # the monthly fit
from scripts.forecast import bias_correction as bc       # the running bias correction
from scripts.forecast import serving_mirror as mirror    # Python copy of the TypeScript serving layer
from scripts.forecast import simulate as sim             # replay helpers
from scripts.forecast import verification_helpers as vh  # plumbing for this notebook only
from scripts.ingest.lib.aqi_utils import compute_subindex, get_aqi_label
from scripts.ingest.lib.config import ist_today

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160, "display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})
COLORS = {"blend": "#2a6fdb", "persistence": "#e07b39", "climatology": "#7a7a7a",
          "actual": "#111111", "served": "#2a6fdb"}
UNITS = {"pm25": "µg/m³", "aqi": "AQI"}
TODAY = ist_today()
T0 = time.time()
# ▶ APP CODE: scripts/forecast/baselines.py · SKILL_* (the rule that picks the held-out years)
print(f"Today (IST): {TODAY}.  The app's model choice is the median over every held-out year Y with "
      f"≥ {B.SKILL_MIN_PRIOR_YEARS} full years (≥ {B.SKILL_FULL_YEAR_MIN_DAYS} valid days) before it, "
      f"a finished season (months {B.SKILL_TEST_MONTHS}), ≥ {B.SKILL_MIN_TEST_DAYS} test days, "
      f"≥ {B.SKILL_MIN_STATIONS_PER_YEAR} qualifying stations per city-year; year range: {B.SKILL_TEST_YEAR_RANGE or 'all'}.")
print(f"Served models: {M.SERVED_MODELS}; horizons with a real forecast: {M.FORECAST_HORIZONS}; "
      f"data ages measured: {M.MODE_DATA_AGES[0]}–{M.MODE_DATA_AGES[-1]} days")

## 1. History, loaded exactly as the monthly refit loads it

The refit (`refit_params.py --source db`) trains on three stacked sources:

| Source | Dates | Where it lives |
|---|---|---|
| XKDR history | 2009 → 1 Sep 2025 | Storage file `xkdr_daily_history.parquet` (local copy in `data/`) |
| OpenAQ S3 archive | 2 Sep 2025 → 16 Aug 2026 | Storage file `openaq_archive_daily_history.parquet` (local copy in `data/`), filling the gap after XKDR ends |
| Our own OpenAQ days | 17 Aug 2026 → today | Supabase `readings_daily`: the recent data the app forecasts from |

`load_history_from_db` reads all three, keeps only **complete** station-days (≥ 12 hours of readings, values 0–2000, the rule in `lib/daily-quality-config.json`), builds the composite AQI (PM2.5 **and** PM10 required), and returns city and station series. It takes a few minutes, mostly paging `readings_daily`, so the result is cached under `data/notebook-cache/` for the rest of the day.

*Note on migration 25:* if `readings_daily.is_complete` does not exist yet, the loader computes completeness from each row's count instead. That is the same rule, applied at read time.

In [ ]:
ro = vh.read_only_client()        # service-role key from .env.local, with every write blocked
if ro.missing_columns:
    print("Migration 25 not run yet: readings_daily.is_complete is missing, so completeness is "
          "computed from each day's count (same rule, applied at read time).")
hist = vh.load_history(ro, refresh=REFRESH_HISTORY)   # ▶ APP CODE: scripts/forecast/baselines.py · load_history_from_db (cached)
for name, frame in hist.items():
    print(f"  {name:<14} {len(frame):>9,} rows   {frame['d'].min():%Y-%m-%d} → {frame['d'].max():%Y-%m-%d}")

In [ ]:
# Daily city means (PM2.5 and AQI), with the three sources shaded.
ERAS = [("XKDR", "2009-01-01", "2025-09-01", "#f2f2f2"),
        ("OpenAQ archive", "2025-09-02", "2026-08-16", "#e6efff"),
        ("readings_daily", "2026-08-17", str(TODAY), "#fff0e0")]

def city_daily(city, pollutant):
    # ▶ APP CODE: scripts/forecast/baselines.py · aqi_series / city_series (what fit_city trains on)
    return B.aqi_series(hist["daily_aqi"], city) if pollutant == "aqi" else B.city_series(hist["daily"], city)

fig, axes = plt.subplots(len(CITIES), 2, figsize=(13, 2.6 * len(CITIES)), squeeze=False, sharex = True)
for i, city in enumerate(CITIES):
    for j, pol in enumerate(["pm25", "aqi"]):
        ax, s = axes[i, j], city_daily(city, pol).loc["2015":]
        for label, a, b, colour in ERAS:
            ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color=colour, zorder=0,
                       label=label if i == 0 and j == 0 else None)
        ax.plot(s.index, s.values, lw=0.5, color="#555")
        ax.plot(s.index, s.rolling(30, min_periods=10).mean(), lw=1.4, color=COLORS["blend"])
        ax.set_xlim(s.first_valid_index(), pd.Timestamp(TODAY))
        ax.set_title(f"{city} · daily city mean {'PM2.5' if pol == 'pm25' else 'AQI'} ({UNITS[pol]})", fontsize=10)
axes[0, 0].legend(loc="upper left", fontsize=8, ncol=3)
plt.tight_layout(); plt.show()

rows = []
for city in CITIES:
    for pol in ["pm25", "aqi"]:
        frame = hist["daily_aqi"] if pol == "aqi" else hist["daily"]
        f = frame[frame[("city" if pol == "aqi" else "city_name")] == city]
        rows.append({"city": city, "pollutant": pol, "first day": f.d.min().date(), "last day": f.d.max().date(),
                     "city-days": len(f), "median stations/day (since Sep 2025)":
                     f[f.d >= "2025-09-01"].n_stations.median()})
pd.DataFrame(rows)

**How to read this.** Grey line: one day. Blue line: 30-day average. Look for (a) the seasonal swing the climatology has to capture, and (b) whether the level jumps where one source hands over to the next (1 Sep 2025, 17 Aug 2026). A jump at a hand-over would mean the sources disagree, and the seasonal normal would be fitted across that disagreement. A city-day counts only when at least 3 stations had a complete day.

## 2. Does the model beat the seasonal average, year after year?

This is the refit's own skill test, `station_skill_years`, **the call `refit_params.fit_city` makes**. Every eligible held-out year Y is scored the same way: train on the years before Y, then forecast Y's pollution season (January and October–December of Y) **per station**. Each year is summarised by the **median station**, and the served number is the **median across years**.

A year Y is tested for a station only if the station has at least `B.SKILL_MIN_PRIOR_YEARS` *full* calendar years of data before Y (a full year has at least `B.SKILL_FULL_YEAR_MIN_DAYS` valid days) and Y's season has finished. A city-year counts only with at least `B.SKILL_MIN_STATIONS_PER_YEAR` qualifying stations. Until Nov 2026 the app used one year, 2024.

Two models are served: **blend** (seasonal normal + a fitted share α of today's departure from normal) and **persistence** (the last reading, unchanged). Climatology is the 0% line.

Two data ages are shown (`AGES_BY_YEAR`): a 1-day-old reading (the median served age, Nov 2026) and a 3-day-old one, a slow week for OpenAQ. Horizon is +1 day. **The dashed line is the median across years: the number the app's label is set from.**

In [ ]:
def station_frame(pol):
    return vh.station_frame(hist, pol)       # the per-station table the refit fits this pollutant on

# All ages and horizons, once: §3 reuses these results.
skill_years = {}
for city in CITIES:
    for pol in POLLUTANTS:
        skill_years[(city, pol)] = B.station_skill_years(station_frame(pol), city)   # ▶ APP CODE: scripts/forecast/baselines.py · station_skill_years (what refit_params.fit_city calls)

# Which years and how many stations each city's median rests on.
elig = []
for (city, pol), r in skill_years.items():
    py = r["per_year"]
    per = py.groupby("test_year").n_stations.max() if not py.empty else pd.Series(dtype=int)
    elig.append({"city": city, "pollutant": pol,
                 "test years": ", ".join(str(y) for y in per.index) or "(none: falls back to the city mean)",
                 "stations per year": " ".join(f"{y % 100:02d}:{n}" for y, n in per.items()),
                 "stations scored (any year)": r["eligible"].station.nunique() if not r["eligible"].empty else 0})
print("Held-out years each city is tested on (year:qualifying stations):")
display(pd.DataFrame(elig))

by_year = pd.concat([r["per_year"].assign(pollutant=pol).rename(columns={"test_year": "year"})
                     for (city, pol), r in skill_years.items() if not r["per_year"].empty], ignore_index=True)
medians = pd.concat([r["summary"].assign(pollutant=pol)
                     for (city, pol), r in skill_years.items() if not r["summary"].empty], ignore_index=True)

ages_2 = sorted(set(AGES_BY_YEAR) | {DATA_AGE_FOR_PLOTS})
fig, axes = plt.subplots(len(CITIES) * len(POLLUTANTS), len(ages_2),
                         figsize=(5.2 * len(ages_2), 2.8 * len(CITIES) * len(POLLUTANTS)), squeeze=False)
for i, (city, pol) in enumerate([(c, p) for c in CITIES for p in POLLUTANTS]):
    for j, age in enumerate(ages_2):
        ax = axes[i, j]
        sel = lambda f: f[(f.city == city) & (f.pollutant == pol) & (f.data_age_days == age) & (f.horizon_days == 1)]
        d, med = sel(by_year), sel(medians)
        for model in M.SERVED_MODELS:
            m = d[d.model == model].sort_values("year")
            ax.plot(m.year, m.skill_pct, "o-", color=COLORS[model], label=model)
            mm = med[med.model == model]
            if len(mm):
                ax.axhline(mm.skill_pct.iat[0], color=COLORS[model], ls="--", lw=1.4,
                           label=f"{model} median {mm.skill_pct.iat[0]:.0f}%")
        ax.axhline(0, color=COLORS["climatology"], lw=1.2, label="climatology (0%)")
        ax.axhline(20, color="green", ls=":", lw=0.8); ax.axhline(5, color="orange", ls=":", lw=0.8)
        ax.set_title(f"{city} {pol} · +1 day from a reading {age} day(s) old", fontsize=10)
        ax.set_ylabel("skill vs seasonal average (%)")
        if len(d):
            ax.set_xticks(sorted(d.year.unique()))
        ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

table2 = by_year[by_year.horizon_days == 1].pivot_table(
    index=["city", "pollutant", "data_age_days", "model"], columns="year", values="skill_pct")
table2["median"] = medians[medians.horizon_days == 1].set_index(
    ["city", "pollutant", "data_age_days", "model"]).skill_pct
display(table2.loc[pd.IndexSlice[:, :, list(ages_2), :], :].round(1))

# The alpha caveat: skill per year uses alpha fitted on years < Y; the app serves alpha fitted on all history.
alpha_rows = []
for (city, pol), r in skill_years.items():
    eff = DATA_AGE_FOR_PLOTS + 1
    per = by_year[(by_year.city == city) & (by_year.pollutant == pol) & (by_year.model == "blend")
                  & (by_year.data_age_days == DATA_AGE_FOR_PLOTS) & (by_year.horizon_days == 1)]
    served = []
    for _, sub in station_frame(pol)[lambda f: f.city == city].groupby("station"):
        s = sub.groupby("d")["v"].mean().asfreq("D").dropna()
        if len(s) >= rp.MIN_HISTORY_DAYS:
            served.append(B.fit_alpha(s, B.climatology(s), eff))   # ▶ APP CODE: scripts/forecast/baselines.py · fit_alpha + climatology (as refit_params.fit_stations fits each station on all history)
    alpha_rows.append({"city": city, "pollutant": pol, "effective horizon": eff,
                       "alpha per test year (median station)": ", ".join(f"{y}:{a:.2f}" for y, a in zip(per.year, per.alpha)),
                       "alpha served (median station, all history)": round(float(np.median(served)), 2) if served else None})
print(f"alpha measured vs alpha served, blend, +1 day from a {DATA_AGE_FOR_PLOTS}-day-old reading:")
display(pd.DataFrame(alpha_rows))

**How to read this.** Above 0 the model beats the seasonal average. Dotted lines mark the app's labels: green 20% (*forecast*), orange 5% (*outlook*). The dashed line is the median the label is set from. If the dots scatter widely around it, the label rests on a noisy measurement; if one year sits far from the rest, the median has kept it from deciding alone. A city tested on only two years (Pune, Nov 2026) has a "median" that is the mean of two years: treat its label as provisional.

**The α table.** Each year's skill was measured with α fitted on the years before it; the app serves α fitted on all history. Differences of a few hundredths change the forecast very little. A large gap would mean the measured skill describes a noticeably different model from the one served.

### Why per station, not the city mean

A city mean is much smoother than any single monitor. Averaging cancels local noise, so a city-mean backtest flatters the forecast. The old notebook scored city means; the app serves stations. The table below puts the two side by side for `REPLAY_YEAR`, +1 day, reading `DATA_AGE_FOR_PLOTS` day(s) old. The audit measured the city-mean error at roughly 71–82% of the per-station error (it looks about 25% better than it is).

In [ ]:
rows = []
for city in CITIES:
    for pol in POLLUTANTS:
        series = city_daily(city, pol)
        cm = B.model_skill_table(series, REPLAY_YEAR, data_ages=(DATA_AGE_FOR_PLOTS,), horizons=(1,))   # ▶ APP CODE: scripts/forecast/baselines.py · model_skill_table (on the city mean)
        st = by_year[(by_year.city == city) & (by_year.pollutant == pol) & (by_year.horizon_days == 1)
                     & (by_year.year == REPLAY_YEAR) & (by_year.data_age_days == DATA_AGE_FOR_PLOTS)]
        if cm.empty or st.empty:
            continue                          # REPLAY_YEAR is not a test year for this city
        for model in M.SERVED_MODELS:
            c, s = cm[cm.model == model].iloc[0], st[st.model == model].iloc[0]
            rows.append({"city": city, "pollutant": pol, "model": model,
                         "city-mean MAE": round(c.mae, 1), "median station MAE": round(s.mae, 1),
                         "city-mean / station": round(c.mae / s.mae, 2),
                         "city-mean skill %": round(c.skill_pct, 1), "station skill %": round(s.skill_pct, 1),
                         "stations scored": int(s.n_stations)})
pd.DataFrame(rows)

### The Pune check: is "persistence at every data age" an artefact of one year?

With the single-year (2024) test, Pune served **persistence at every data age**, beating the seasonal average by 15–23% even when the reading was 7 days old. The cell below shows Pune's winner (+1 day, ages 0–7) for each test year and for the median.

Two things to watch: **how many stations** the median rests on (Pune's stations mostly started in 2020–2023 with patchy years, so only two qualify, for 2024 and 2025); and the **climatology error itself**. If the seasonal normal is poor (short or shifting history), even an old reading beats it. That would say more about the climatology than about persistence.

In [ ]:
if "Pune" in CITIES:
    pune = []
    for pol in POLLUTANTS:
        r = skill_years[("Pune", pol)]
        if r["per_year"].empty:
            continue
        for year, t in r["per_year"][lambda f: f.horizon_days == 1].groupby("test_year"):
            best = B._pick_served_model(t)        # ▶ APP CODE: scripts/forecast/baselines.py · _pick_served_model (how the served model is chosen)
            pune.append(best.assign(pollutant=pol, year=str(year)))
        pune.append(r["modes"][lambda f: f.horizon_days == 1].assign(pollutant=pol, year="median (served)"))
    pune = pd.concat(pune, ignore_index=True)
    pune["winner"] = pune.model.str[:5] + " " + pune.skill_pct.round(0).astype(int).astype(str) + "%"
    display(pune.pivot_table(index=["pollutant", "data_age_days"], columns="year", values="winner", aggfunc="first"))
    display(pune.groupby(["pollutant", "year"]).agg(stations=("n_stations", "first"),
                                                    climatology_MAE=("clim_mae", "first")).round(1).T)
    always = pune.groupby(["pollutant", "year"]).model.apply(lambda m: (m == "persistence").all())
    for (pol, year), yes in always.items():
        print(f"  Pune {pol} {year}: persistence wins at every age? {'YES' if yes else 'no'}")
else:
    print("Pune is not in CITIES; add it to run this check.")

**How to read this.** Each cell is the winning model (first 5 letters) and its skill at that data age (rows) for that held-out year (columns); the last column is what the app serves. If persistence wins at every age in **every** year, it is a property of Pune's stations: their seasonal normal does not describe them well. If it happens only in one year, the choice rests on an unusual season. Check the stations row too: a median over 2 stations is fragile.

## 3. How skill decays as the input reading ages: the table that decides what is served

This is `station_skill_years(...)["modes"]`, **what `refit_params.fit_city` stores in `forecast_modes`**, from the same results as §2. For every data age (0–7 days) and horizon (+1, +2, +3 days) it takes both served models' **median skill across the held-out years** (each year itself the median station), keeps the better one, and sets the label from its skill: ≥ 20% *forecast*, ≥ 5% *outlook*, otherwise *seasonal_normal*. The bands are pooled across years (each station's relative errors from all its test years together, then the median station).

An input `age` days old forecasting `h` days ahead is really an `(age + h)`-day problem, and the blend's α is fitted for exactly that effective horizon (`model.alpha_for`).

**Which row matters most.** Amrita's check of the served forecasts (Nov 2026): the **median served data age is 1 day**. So the age-1 row (outlined in the heatmaps) is what most users see; ages 2–5 are what they see in a slow OpenAQ week.

In [ ]:
mode_tables = {k: r["modes"] for k, r in skill_years.items()}      # ▶ APP CODE: station_skill_years(...)["modes"] (what refit_params.fit_city stores)
skill_tables = {k: r["summary"] for k, r in skill_years.items()}    # ▶ APP CODE: station_skill_years(...)["summary"] (both models, median over years)

keys = [k for k in mode_tables if not mode_tables[k].empty]
fig, axes = plt.subplots(len(keys), 3, figsize=(14, 3 * len(keys)), squeeze=False, sharey="row")
for i, (city, pol) in enumerate(keys):
    sk, served = skill_tables[(city, pol)], mode_tables[(city, pol)]
    for j, h in enumerate(M.FORECAST_HORIZONS):
        ax = axes[i, j]
        for model in M.SERVED_MODELS:
            m = sk[(sk.model == model) & (sk.horizon_days == h)].sort_values("data_age_days")
            ax.plot(m.data_age_days, m.skill_pct, "o-", ms=3, color=COLORS[model], label=model)
        s = served[served.horizon_days == h]
        ax.scatter(s.data_age_days, s.skill_pct, s=70, facecolors="none", edgecolors="k", lw=1.2,
                   label="served (the winner)", zorder=3)
        ax.axhspan(20, max(60, sk.skill_pct.max() + 5), color="#e5f5e5", zorder=0)
        ax.axhspan(5, 20, color="#fff4dd", zorder=0)
        ax.axhline(0, color=COLORS["climatology"], lw=1.2)
        ax.set_title(f"{city} {pol} · +{h} day (median of {served.n_years.iat[0]} years)", fontsize=10)
        ax.set_xlabel("age of input reading (days)"); ax.set_xticks(M.MODE_DATA_AGES)
    axes[i, 0].set_ylabel("skill vs seasonal avg (%)")
axes[0, 0].legend(fontsize=8)
plt.suptitle("Median skill across held-out years.  Green band: labelled 'forecast' (≥20%).  Amber: 'outlook' (5–20%).  Below: seasonal normal.", fontsize=10)
plt.tight_layout(); plt.show()

# ── Heatmaps of the served table: one small multiple per city and pollutant ──
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.patches import Rectangle

# Diverging: red = worse than the seasonal average, grey = no better, blue = better.
SKILL_CMAP = LinearSegmentedColormap.from_list(
    "skill", [(0.0, "#b2182b"), (0.5, "#f0efec"), (0.8, "#2a6fdb"), (1.0, "#123f8c")], N=256)   # grey exactly at 0
SKILL_NORM = TwoSlopeNorm(vmin=-30, vcenter=0, vmax=60)      # values beyond are clipped to the end colours
LABEL = {"forecast": "F", "outlook": "O", "seasonal_normal": "—"}

ncol = len(POLLUTANTS)
nrow = len(CITIES)
fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol + 1.5, 3.6 * nrow + 0.8), squeeze=False,
                         layout="constrained")
for i, city in enumerate(CITIES):
    for j, pol in enumerate(POLLUTANTS):
        ax = axes[i, j]
        t = mode_tables.get((city, pol), pd.DataFrame())
        ax.grid(False)
        if t.empty:
            ax.set_axis_off(); ax.set_title(f"{city} {pol}: no eligible year", fontsize=9)
            continue
        grid = t.pivot(index="data_age_days", columns="horizon_days", values="skill_pct").reindex(
            index=list(M.MODE_DATA_AGES), columns=list(M.FORECAST_HORIZONS))
        im = ax.imshow(grid.values, cmap=SKILL_CMAP, norm=SKILL_NORM, aspect="auto")
        cell = t.set_index(["data_age_days", "horizon_days"])
        for r, age in enumerate(grid.index):
            for c, h in enumerate(grid.columns):
                if (age, h) not in cell.index:
                    continue
                x = cell.loc[(age, h)]
                strong = abs(SKILL_NORM(x.skill_pct) - 0.5) > 0.32          # white text on the dark ends
                ax.text(c, r, f"{x.model[0].upper()} {round(x.skill_pct) + 0:.0f}%  {LABEL[x['mode']]}",
                        ha="center", va="center", fontsize=8.5, color="white" if strong else "#1a1a19")
        if DATA_AGE_FOR_PLOTS in grid.index:                                 # outline the median served age
            r = list(grid.index).index(DATA_AGE_FOR_PLOTS)
            ax.add_patch(Rectangle((-0.5, r - 0.5), len(grid.columns), 1, fill=False, lw=2, ec="#1a1a19"))
        ax.set_xticks(range(len(grid.columns))); ax.set_xticklabels([f"+{h} d" for h in grid.columns])
        ax.set_yticks(range(len(grid.index))); ax.set_yticklabels(grid.index)
        ax.set_xlabel("horizon"); ax.set_ylabel("age of input reading (days)")
        yrs = t.test_years.iat[0].split(",")
        ax.set_title(f"{city} {pol} · median of {len(yrs)} test years ({yrs[0]}–{yrs[-1]})", fontsize=9)
cb = fig.colorbar(im, ax=axes, shrink=min(0.6, 3 / nrow + 0.2), pad=0.02)
cb.set_label("median skill vs seasonal average (%)")
# The label thresholds, marked on the colour bar itself.
cb.set_ticks([-30, -15, 0, B.MIN_USEFUL_SKILL_PCT, 20, 40, 60])
cb.set_ticklabels(["≤ −30", "−15", "0", f"{B.MIN_USEFUL_SKILL_PCT:.0f}  outlook", "20  forecast", "40", "≥ 60"])
for level in (B.MIN_USEFUL_SKILL_PCT, 20):
    cb.ax.axhline(level, color="#1a1a19", lw=1.5)
fig.suptitle("Served model and label by data age × horizon\nCell: model (B blend / P persistence), median skill, "
             "label (F forecast / O outlook / — seasonal normal).  Outlined row: the median served data age "
             f"({DATA_AGE_FOR_PLOTS} day).", fontsize=10)
plt.show()

**How to read this.** Each line should fall as the input ages; that fall is the decay. In the heatmaps, blue cells beat the seasonal average and red cells are worse than it; grey is "no better". The two black marks on the colour bar are the label thresholds: below 5% the app stops calling the number a forecast and serves the seasonal normal instead (—). Read the outlined row first: it is the median data age actually served (1 day, Nov 2026). §9 shows the data ages actually served over the last nights.

### Does the stored table match? (`forecast_modes` in Supabase)

The refit wrote `forecast_modes` from this same function. If it ran with today's code on the same history, the table above and the stored one agree row for row. If `model` is empty in the stored rows, they were written by the **pre-audit** refit, which measured only the blend. If the stored table was written before the multi-year test (Nov 2026), it was measured on 2024 alone and skill will differ until the next refit.

In [ ]:
modes_live = nf.load_modes(ro)        # ▶ APP CODE: scripts/forecast/nightly_forecast.py · load_modes (what nightly reads)
params_live = nf.load_params(ro)      # ▶ APP CODE: scripts/forecast/nightly_forecast.py · load_params
status = vh.which_code_wrote(params_live, modes_live)
print(f"forecast_modes rows with a model: {status['modes_with_model']}/{status['modes_total']};  "
      f"forecast_params rows with per-horizon alphas: {status['params_with_alphas']}/{status['params_total']}")
print("Stored tables were written by the", "NEW (post-audit) refit." if status["new_code"] else
      "OLD (pre-audit) refit: differences below are expected until the new refit runs on main.")

rows = []
for (city, pol), t in mode_tables.items():
    for r in t.itertuples():
        live = modes_live.get((city, pol, int(r.data_age_days), int(r.horizon_days)))
        rows.append({"city": city, "pollutant": pol, "age": r.data_age_days, "h": r.horizon_days,
                     "model (recomputed)": r.model, "model (stored)": live.get("model") if live else "(no row)",
                     "mode (recomputed)": r.mode, "mode (stored)": live["mode"] if live else "(no row)",
                     "skill % (recomputed)": round(r.skill_pct, 1),
                     "skill % (stored)": round(live["skill_pct"], 1) if live else None})
cmp3 = pd.DataFrame(rows)
cmp3["same"] = (cmp3["mode (recomputed)"] == cmp3["mode (stored)"]) & (
    (cmp3["skill % (recomputed)"] - cmp3["skill % (stored)"]).abs() < 0.5)
print(f"rows that agree (same label, skill within 0.5 points): {cmp3.same.sum()}/{len(cmp3)}")
cmp3[~cmp3.same].head(24)

## 4. Forecast vs what actually happened, using the serving function

Here the number scored is **the number the app would have served**, in two steps: `served` is the model output from the serving function, and `served_bc` is that number after nightly's **bias correction**, replayed night by night just below. `served_bc` is what would have been on screen. For each night of the held-out season, every station's forecast comes from `nightly_forecast.build_station_rows`, the function the nightly job runs. It uses:

- parameters (climatology, α for every effective horizon, history length) fitted by the refit's own `fit_city` + `fit_stations` **on the years before `REPLAY_YEAR` only**, which is what the refit would have stored on 1 January of that year;
- the mode table from §3 (which model to serve at each age, and its label). That table is the median over all held-out years, including years after `REPLAY_YEAR`, so the *choice* of model has seen later seasons. This section checks the serving arithmetic, not that choice;
- an input reading exactly `DATA_AGE_FOR_PLOTS` days old.

> **Limitation.** The live app fits parameters on all history. Using all history here would let the forecast see the season it is being scored on, so this section deliberately refits on earlier years. The serving arithmetic is unchanged. Nights whose reading is missing exactly `DATA_AGE_FOR_PLOTS` days earlier are skipped. The app would use an older reading instead, which §3 already scores.

In [ ]:
targets = vh.season_dates(REPLAY_YEAR)     # Jan + Oct–Dec of the held-out year
replays, params_test, city_fit_test = {}, {}, {}
for city in CITIES:
    for pol in POLLUTANTS:
        # The city fit is kept too: §4 Check E compares it with the all-history fit.
        city_fit_test[(city, pol)], params_test[(city, pol)] = vh.fit_as_of(hist, city, pol, REPLAY_YEAR)   # ▶ APP CODE: refit_params.fit_city + fit_stations (inside the helper)
        modes = vh.modes_index(mode_tables[(city, pol)], city, pol)
        replays[(city, pol)] = vh.replay_served(                                       # ▶ APP CODE: nightly_forecast.build_station_rows (inside the helper)
            station_frame(pol), city, pol, params_test[(city, pol)], modes, targets, DATA_AGE_FOR_PLOTS)
        r = replays[(city, pol)]
        print(f"{city} {pol}: {r.station.nunique()} stations, {len(r):,} served station-days; "
              f"labels: {r['mode'].value_counts().to_dict()}")

### The bias correction, replayed: the number that would have been on screen

Production nightly does one more thing after `build_station_rows`: a city-level **bias correction** (§5 explains it). Without replaying it, "served" above would be the model output, not the number on screen. The cell below replays it **night by night with nightly's own functions**:

1. the history is the replay's own +1-day forecasts of the last `nf.BIAS_HISTORY_DAYS` days (nightly's `load_scored_history` window);
2. `bias_correction.city_day_errors` turns them into one mean error per day for the city, learning only from rows labelled *forecast*/*outlook*;
3. `bias_correction.decide_all` decides apply/skip and the offset;
4. `bias_correction.apply_corrections` corrects that night's *forecast*/*outlook* rows; *seasonal_normal* rows are left alone (`BIAS_CORRECT_MODES`, since commit e28575c).

**No look-ahead.** On night N only errors for days up to N − `DATA_AGE_FOR_PLOTS` are used: a reading that old means nothing newer had been published, so neither had any newer actual value. To give the correction a history at the start of each season, the replay also forecasts the ~3 weeks before each stretch of held-out days (late September, and December of the year before), at +1 day only.

> **What cannot be reconstructed honestly.**
> - **Mixed data ages.** Live, the history mixes forecasts made from readings of different ages and stations that dropped in and out. The replay's history is only its own forecasts at exactly `DATA_AGE_FOR_PLOTS` days, for stations that had a reading that old.
> - **Burst publishing.** The replay assumes every station's day up to N − `DATA_AGE_FOR_PLOTS` was known on night N. Live, OpenAQ publishes in bursts, so some stations' days arrive later and the city's daily mean error rests on fewer stations.
> - **The January warm-up is in-sample.** December of the year before `REPLAY_YEAR` was in the training data, so its errors (the correction's history for early January) are a little smaller than they would have been live. The October warm-up (September of `REPLAY_YEAR`) is out-of-sample.
>
> So the "after" numbers show what the correction *would do* with a clean history; the live record is §5 and §9.

In [ ]:
# Warm-up days: the +1-day history the correction needs at the start of each held-out stretch.
warm_dates = vh.bias_warmup_dates(targets, nf.BIAS_HISTORY_DAYS, max(M.FORECAST_HORIZONS))
bias_logs = {}
for (city, pol), rep in replays.items():
    modes = vh.modes_index(mode_tables[(city, pol)], city, pol)
    warm = vh.replay_served(station_frame(pol), city, pol, params_test[(city, pol)], modes,   # ▶ APP CODE: nightly_forecast.build_station_rows (inside the helper)
                            warm_dates, DATA_AGE_FOR_PLOTS, horizons=(1,))
    # Adds `served_bc` (after correction) and `bias_applied` to each replayed row.
    replays[(city, pol)], bias_logs[(city, pol)] = vh.replay_bias_correction(   # ▶ APP CODE: bias_correction.city_day_errors + decide_all + apply_corrections (inside the helper)
        rep, warm, city, pol, DATA_AGE_FOR_PLOTS)
    log = bias_logs[(city, pol)]
    print(f"{city} {pol}: correction applied on {int(log.applied.sum())} of {len(log)} nights; "
          f"median offset when applied {log[log.applied].offset.median() if log.applied.any() else float('nan'):+.1f} {UNITS[pol]}")

def _err(g, col):
    return (g[col] - g.actual)

rows = []
for (city, pol), rep in replays.items():
    for (h, mode), g in rep[rep.actual.notna()].groupby(["horizon", "mode"]):
        mae_raw, mae_bc = _err(g, "served").abs().mean(), _err(g, "served_bc").abs().mean()
        clim = (g.climatology - g.actual).abs().mean()
        rows.append({"city": city, "pollutant": pol, "h": h, "label": mode, "station-days": len(g),
                     "rows corrected %": 100 * g.bias_applied.mean(),
                     "mean error before": _err(g, "served").mean(), "mean error after": _err(g, "served_bc").mean(),
                     "MAE before": mae_raw, "MAE after (as served)": mae_bc,
                     "skill before %": 100 * (clim - mae_raw) / clim, "skill after %": 100 * (clim - mae_bc) / clim})
bias_replay = pd.DataFrame(rows)
print(f"\n{REPLAY_YEAR} season, reading {DATA_AGE_FOR_PLOTS} d old. Error = served − actual (negative: under-forecast). "
      "Skill is vs the seasonal normal on the same station-days.")
bias_replay.round(1)

**How to read this.** *Before* is the model output (what Check A compares with the refit); *after* is the number the app would have shown. If "MAE after" is below "MAE before", the correction helped in this season; the self-policing in `decide` should keep it switched off where it does not. *seasonal_normal* rows must show 0% corrected and identical before/after. A *mean error* that moves towards 0 after correction is the correction doing its job (removing a lag); a lower MAE is what actually matters to a user.

### Check A: is the label honest? Served-number error vs the error the refit measured

The refit's skill (§3) was measured with `model_skill_table` on each station. If serving really does the same arithmetic, scoring the **served** numbers (before bias correction: the correction is a separate step the refit never measured) must give the same error for every station that gets a real forecast. Any gap would mean the label describes a different number from the one on screen, which was the audit's §7.2 finding about the old code.

In [ ]:
rows = []
for (city, pol), rep in replays.items():
    served_model = mode_tables[(city, pol)].set_index(["data_age_days", "horizon_days"])
    for station, sub in station_frame(pol)[lambda f: f.city == city].groupby("station"):
        s = sub.groupby("d")["v"].mean().asfreq("D")
        t = B.model_skill_table(s, REPLAY_YEAR, data_ages=(DATA_AGE_FOR_PLOTS,))   # ▶ APP CODE: scripts/forecast/baselines.py · model_skill_table (per station, as station_model_skill runs it)
        if t.empty:
            continue
        for h in M.FORECAST_HORIZONS:
            row = served_model.loc[(DATA_AGE_FOR_PLOTS, h)]
            if row["mode"] == "seasonal_normal":
                continue                          # served as the seasonal normal; no model to compare
            measured = t[(t.horizon_days == h) & (t.model == row.model)]
            got = rep[(rep.station == station) & (rep.horizon == h) & rep.actual.notna()]
            if measured.empty or got.empty:
                continue
            rows.append({"city": city, "pollutant": pol, "station": station[:8], "h": h, "model": row.model,
                         "measured MAE (refit)": measured.mae.iat[0],
                         "served MAE (build_station_rows)": float((got.served - got.actual).abs().mean()),
                         "served label": got["mode"].mode().iat[0]})
checkA = pd.DataFrame(rows)
if checkA.empty:
    print("No station had a real forecast at this data age; nothing to compare.")
else:
    checkA["difference"] = checkA["served MAE (build_station_rows)"] - checkA["measured MAE (refit)"]
    ok = checkA.difference.abs() < 0.05      # served values are rounded to 0.01
    print(f"stations whose served error equals the measured error (within 0.05): {ok.sum()}/{len(checkA)}")
    display(checkA.round(2).sort_values("difference", key=abs, ascending=False).head(10))

**How to read this.** "Within 0.05" for (nearly) every station means the label and band shown in the app describe the number served. A station that differs is usually one with under a year of its own history before the test year: serving gives it the seasonal normal (`served label` = seasonal_normal) while the skill table never scored it.

### Check B: the plots

Top: the city average of the served +1-day forecasts (solid: after bias correction, as shown; dotted: before) against the city average of what stations measured, with the seasonal normal for reference (Oct–Dec of the held-out year). Below: three randomly picked stations (fixed seed).

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
for (city, pol), rep in replays.items():
    r1 = rep[(rep.horizon == 1) & (rep.target >= f"{REPLAY_YEAR}-10-01")]
    if r1.empty:
        continue
    real = r1[r1["mode"] != "seasonal_normal"].station.unique()
    picks = list(rng.choice(real, size=min(3, len(real)), replace=False)) if len(real) else []
    fig, axes = plt.subplots(1 + len(picks), 1, figsize=(12, 2.8 * (1 + len(picks))), squeeze=False)
    # asfreq("D") puts missing days back in, so gaps show as gaps instead of joined lines.
    city_mean = r1.groupby("target")[["served", "served_bc", "actual", "climatology"]].mean().asfreq("D")
    title = (f"{city} {pol} · city average over {r1.station.nunique()} stations "
             f"({len(real)} with a real forecast; the rest serve the seasonal normal)")
    panels = [(title, city_mean)] + [
        (f"station {p[:8]}… ({vh.manifest_name(p)})", r1[r1.station == p].set_index("target").asfreq("D")) for p in picks]
    for ax, (title, d) in zip(axes[:, 0], panels):
        ax.plot(d.index, d.actual, color=COLORS["actual"], lw=1.4, label="actual (complete days)")
        ax.plot(d.index, d.served_bc, color=COLORS["served"], lw=1.2, label=f"served +1 day (reading {DATA_AGE_FOR_PLOTS} d old)")
        ax.plot(d.index, d.served, color=COLORS["served"], lw=0.8, ls=":", label="before bias correction")
        ax.plot(d.index, d.climatology, color=COLORS["climatology"], ls="--", lw=1, label="seasonal normal")
        ax.set_title(title, fontsize=10); ax.set_ylabel(UNITS[pol])
    axes[0, 0].legend(fontsize=8, ncol=3)
    plt.tight_layout(); plt.show()

### Check C: is the right health band shown?

A user acts on the category (Good, Satisfactory, Moderate, Poor, Very Poor, Severe), not the number. This scores how often the band of the served +1-day number matches the band of what happened, per station, using the app's own band lookup (`compute_subindex` → `get_aqi_label`, the Python port of `lib/aqi-utils.ts`).

In [ ]:
def band(pol, v):
    index = v if pol == "aqi" else compute_subindex(pol, v)   # ▶ APP CODE: scripts/ingest/lib/aqi_utils.py · compute_subindex
    return get_aqi_label(index)                               # ▶ APP CODE: scripts/ingest/lib/aqi_utils.py · get_aqi_label

ORDER = {"Good": 0, "Satisfactory": 1, "Moderate": 2, "Poor": 3, "Very Poor": 4, "Severe": 5}
rows = []
for (city, pol), rep in replays.items():
    r1 = rep[(rep.horizon == 1) & rep.actual.notna()].copy()
    for col in ("served", "served_bc", "actual", "climatology"):
        r1[col + "_band"] = [ORDER.get(band(pol, v), np.nan) for v in r1[col]]
    for who in ("served_bc", "served", "climatology"):
        gap = (r1[who + "_band"] - r1["actual_band"]).abs()
        rows.append({"city": city, "pollutant": pol,
                     "what": {"served_bc": "served (as shown)", "served": "before bias correction"}.get(who, who), "station-days": len(r1),
                     "same band %": round(100 * (gap == 0).mean(), 1),
                     "within one band %": round(100 * (gap <= 1).mean(), 1)})
pd.DataFrame(rows)

### Check D: does the band hold 80% of what happened?

Each *forecast*/*outlook* day in the app carries a band, `value × (1 ± band_p80)` (`lib/api/forecast.ts`, here through its Python mirror `serving_mirror.expand_day`). `band_p80` comes from the mode table (§3, the same table and model the replay serves): the 80th percentile of the forecast's relative error, pooled over each station's held-out years, median station. If it is honest, about **80%** of actual days fall inside it.

This scores that on the replay, by city, pollutant, label and horizon, around the number as shown (after bias correction) and before correction. *seasonal_normal* rows carry no band and are not scored.

> **Limitations.** (1) `band_p80` is the **median station's** p80, applied to every station: noisy stations will fall outside more often, quiet ones less, so the pooled share need not be exactly 80% even if the band is well measured. (2) The band was pooled over all held-out years, **including `REPLAY_YEAR`**, so this season is partly in its own band: expect coverage slightly flattering. (3) The band measures the raw model's errors; the bias correction moves the centre afterwards, so "as served" can differ from "before".

In [ ]:
TARGET_COVERAGE_PCT = 80       # band_p80 is meant to contain 80% of what happens

rows = []
for (city, pol), rep in replays.items():
    banded = rep[rep.band_p80.notna() & rep.actual.notna()].copy()
    if banded.empty:
        continue
    for col in ("served_bc", "served"):
        b = vh.served_band(banded, city, pol, value_col=col)   # ▶ APP CODE (mirror of lib/api/forecast.ts): expandDay's band_low / band_high (inside the helper)
        banded[f"inside_{col}"] = (banded.actual >= b.band_low) & (banded.actual <= b.band_high)
    for (mode, h), g in banded.groupby(["mode", "horizon"]):
        rows.append({"city": city, "pollutant": pol, "label": mode, "h": h,
                     "station-days": len(g), "stations": g.station.nunique(),
                     "band_p80 (± share of value)": g.band_p80.astype(float).median(),
                     "inside band % (as served)": 100 * g.inside_served_bc.mean(),
                     "inside band % (before bias corr.)": 100 * g.inside_served.mean()})
coverage = pd.DataFrame(rows)
if coverage.empty:
    print("No replayed row carried a band (every row was seasonal_normal at this data age).")
else:
    coverage["vs 80% target (points)"] = coverage["inside band % (as served)"] - TARGET_COVERAGE_PCT
    print(f"Share of actual days inside the served band, {REPLAY_YEAR} season, reading {DATA_AGE_FOR_PLOTS} d old "
          f"(target {TARGET_COVERAGE_PCT}%):")
    display(coverage.round(2))

# Is the band in that table the one the app serves? Recomputed (§3) vs stored forecast_modes, same data age.
band_cmp = []
for (city, pol), t in mode_tables.items():
    for r in t[t.data_age_days == DATA_AGE_FOR_PLOTS].itertuples():
        live = modes_live.get((city, pol, int(r.data_age_days), int(r.horizon_days)))   # from nightly_forecast.load_modes (§3)
        band_cmp.append({"city": city, "pollutant": pol, "h": r.horizon_days, "label": r.mode,
                         "band_p80 recomputed": r.band_p80,
                         "band_p80 stored": live.get("band_p80") if live else None})
print("band_p80 used above vs the one stored in forecast_modes (differs until the refit runs on today's code and history):")
pd.DataFrame(band_cmp).round(3)

**How to read this.** "inside band % (as served)" near 80 means the band on screen is honest. Well below 80 (say under 70) means the app's band is too narrow: users are told the range is tighter than it is. Well above (over 90) means it is wider than it needs to be. Read it per label: *outlook* is meant to be the less certain one, but its band is measured the same way, so it should also hold about 80%. Because of limitation (2), a value a few points above 80 here is not reassuring by itself; a value clearly below 80 is a real warning.

### Check E: how different is the fit the app serves from the fit scored here?

The app serves parameters fitted on **all history** (`refit_params.py`, monthly). Everything this notebook scores — skill (§2, §3), the replay (§4, §7) and the bands — uses fits on **years before the test year**, on purpose: a fit that has seen the season it is scored on would look better than it is. That is the right methodology, and it is not changed here. But it means the measured quality describes a slightly **different set of numbers** from the ones served.

This check measures that difference for `REPLAY_YEAR`, with the refit's own `fit_city` + `fit_stations` run twice: on all history (what the app serves) and on years before `REPLAY_YEAR` (what §4 scores):

- **α (the share of today's departure from normal carried forward)**, per city, pollutant and *effective horizon* (data age + days ahead). The city α, and the median and 90th-percentile station difference, for stations with enough of their own history in both fits.
- **The seasonal normal (climatology curve)**, per station, as the mean absolute difference over the pollution season, in the pollutant's unit (µg/m³ for PM2.5, AQI points for AQI) and as a % of the served curve.
- **Stations new since `REPLAY_YEAR`**: no history of their own before it, so §4 scored them on the city curve, while the app now serves their own. Their quality is not measured by §4 at all.

In [ ]:
eff_horizons = [DATA_AGE_FOR_PLOTS + h for h in M.FORECAST_HORIZONS]   # the α's the §4 replay actually used
# refit_params.fit_stations gives a station its OWN seasonal curve from 60 days of history (the city curve below that).
OWN_CURVE_MIN_DAYS = 60
alpha_rows, clim_rows = [], []
for city in CITIES:
    for pol in POLLUTANTS:
        city_all, params_all = vh.fit_as_of(hist, city, pol, None)    # ▶ APP CODE: refit_params.fit_city + fit_stations on ALL history (what the monthly refit stores; inside the helper)
        city_old, params_old = city_fit_test.get((city, pol)), params_test.get((city, pol), {})
        if city_all is None or city_old is None:
            continue
        diff = vh.fit_difference(params_all, params_old)                         # plain subtraction, per station (season days)
        diff_year = vh.fit_difference(params_all, params_old, season_only=False)  # the same over the whole year
        if diff.empty:
            continue
        own = diff[diff.own_fit_served & diff.own_fit_scored]                    # own α in both fits
        curve = lambda f: f[(f.history_days_served >= OWN_CURVE_MIN_DAYS) & (f.history_days_scored >= OWN_CURVE_MIN_DAYS)]
        own_curve, own_curve_year = curve(diff), curve(diff_year)                # own seasonal curve in both fits
        # Sanity check: is the all-history fit what is stored? (Differs if the stored refit used older history or older code.)
        stored_gap = [abs(params_all[m]["alpha_h1"] - params_live[(m, pol)]["alpha_h1"])
                      for m in params_all if (m, pol) in params_live and params_live[(m, pol)].get("alpha_h1") is not None]
        for k in M.EFFECTIVE_HORIZONS:
            col = f"alpha_diff_h{k}"
            d = own[col].abs() if col in own else pd.Series(dtype=float)
            alpha_rows.append({"city": city, "pollutant": pol, "effective horizon": k,
                               "city α served (all history)": city_all["alphas"][k],
                               f"city α scored (< {REPLAY_YEAR})": city_old["alphas"][k],
                               "city Δα": city_all["alphas"][k] - city_old["alphas"][k],
                               "stations (own α in both)": len(d),
                               "station |Δα| median": d.median(), "station |Δα| 90th pct": d.quantile(0.9)})
        season_doys = sorted({t.dayofyear for t in targets})
        ca = city_all["clim"].reindex(season_doys); co = city_old["clim"].reindex(season_doys)
        clim_rows.append({"city": city, "pollutant": pol, "unit": UNITS[pol],
                          "city curve: mean |Δ| (season)": (ca - co).abs().mean(),
                          "city curve: mean |Δ| % (season)": 100 * ((ca - co).abs() / ca).mean(),
                          "stations (own curve in both)": len(own_curve),
                          "station curve: median |Δ| (season)": own_curve.clim_mae.median(),
                          "station curve: median |Δ| % (season)": own_curve.clim_pct.median(),
                          "station curve: 90th pct |Δ| % (season)": own_curve.clim_pct.quantile(0.9),
                          "station curve: median |Δ| % (whole year)": own_curve_year.clim_pct.median(),
                          f"stations new since {REPLAY_YEAR} (own fit now, none before)": int((diff.own_fit_served & ~diff.own_fit_scored).sum()),
                          "max |α_h1 − stored α_h1|": max(stored_gap) if stored_gap else None})
alpha_diff = pd.DataFrame(alpha_rows)
clim_diff = pd.DataFrame(clim_rows)

print("Median station |Δα| (all-history fit − pre-REPLAY_YEAR fit), by effective horizon:")
display(alpha_diff.pivot_table(index=["city", "pollutant"], columns="effective horizon", values="station |Δα| median").round(3))
print(f"α in detail, at the effective horizons the §4 replay used ({eff_horizons}):")
display(alpha_diff[alpha_diff["effective horizon"].isin(eff_horizons)].round(3))
print("Seasonal normal: all-history curve vs pre-REPLAY_YEAR curve:")
clim_diff.round(2)

**How to read this — in plain language.** The skill, labels and bands above were measured on a model fitted *without* the season it was tested on. The app serves the same model refitted on *all* years. **The measured quality transfers to what the app serves only if those two fits are close.** These tables say how close.

- **α.** A Δα of 0.05 means the served forecast carries 5 percentage points more (or less) of today's departure from normal. With a departure of 60 µg/m³ that moves the forecast by 3 µg/m³, small next to a typical daily error of 20–50. As a rule of thumb, station |Δα| under about 0.05 is negligible; above about 0.15, the served blend behaves noticeably differently from the scored one and its skill is not well described by §3.
- **Seasonal normal.** A season-average difference of a few % means the climatology the forecast starts from, and the seasonal normal it is judged against, are essentially the same curve. Over about 10% means adding the newer years shifted the curve (for example, a city that got cleaner or dirtier), and both the served seasonal normal and the skill measured against the older curve should be read with that in mind.
- **New stations.** Stations counted as "new since `REPLAY_YEAR`" are served on their own fit but were scored on the city curve, or not at all. Their quality is not covered by §4; §3's skill covers them only for years where they qualified.
- **Last column.** The all-history fit here should match what is stored in `forecast_params`. A large gap means the stored refit used different history or older code (§3 and §8 say which), not a problem with this check.

These are rules of thumb for reading the table, not pass/fail tests.

## 5. Bias correction: what it changed, and whether it helped

After the model, nightly applies a city-level offset learned from the last three weeks of its own +1-day errors (`bias_correction.py`). It is switched on per city only when it has been helping, and even then it only touches rows labelled *forecast* or *outlook* (`bias_correction.BIAS_CORRECT_MODES`). Since Nov 2026 it also **learns only from those rows**: a past +1-day row labelled *seasonal_normal* carries the climatology's error, not the model's, so it is left out of both the offset and the apply/skip decision. A city with no such rows gets an explicit *skip* with that reason (`bias_correction.decide_all`). Rows labelled *seasonal_normal* are left as the plain seasonal average, because that label says "typical for this time of year": shifting it by recent errors would make it a forecast it does not claim to be. Stored rows keep the uncorrected number in `forecast_daily.value_raw`, and the decision in `forecast_bias`.

This section reads the last `nf.BIAS_HISTORY_DAYS` days of served +1-day forecasts with nightly's own loader. It scores the corrected and uncorrected values against complete days, then re-runs the decision with today's `decide` (which scores PM2.5 against PM2.5; the pre-audit code scored PM2.5 against AQI, audit §7.1).

In [ ]:
city_of = vh.city_of_monitor()
print("Rows the correction may touch, by label:", bc.BIAS_CORRECT_MODES)   # ▶ APP CODE: scripts/forecast/bias_correction.py · BIAS_CORRECT_MODES
since = TODAY - timedelta(days=nf.BIAS_HISTORY_DAYS)
fc_hist, actuals = nf.load_scored_history(ro, POLLUTANTS, since)        # ▶ APP CODE: scripts/forecast/nightly_forecast.py · load_scored_history
print(f"{len(fc_hist):,} served +1-day forecasts since {since}; {len(actuals):,} complete station-days to score against")

stored_bias = vh.fetch_all(ro, "forecast_bias")
stored_bias = stored_bias[stored_bias.city.isin(CITIES) & stored_bias.pollutant.isin(POLLUTANTS)]
print("\nWhat the last nightly run decided (forecast_bias):")
display(stored_bias[["city", "pollutant", "applied", "offset_value", "mae_raw", "mae_corrected", "reason", "computed_at"]])

fc_df = pd.DataFrame(fc_hist)
fc_df["city"] = fc_df.monitor_id.map(city_of)
fc_df = fc_df[fc_df.city.isin(CITIES)]
fc_df["actual"] = [actuals.get((m, d, p)) for m, d, p in zip(fc_df.monitor_id, fc_df.target_date, fc_df.pollutant)]
fc_df["raw"] = fc_df.value_raw.fillna(fc_df.value)
scored = fc_df.dropna(subset=["actual"])
summary = (scored.groupby(["city", "pollutant"])
                 .apply(lambda g: pd.Series({
                     "station-days": len(g),
                     "days corrected": int(g.value_raw.notna().sum()),
                     "MAE before correction": (g.raw - g.actual).abs().mean(),
                     "MAE as served": (g.value - g.actual).abs().mean()})).round(1))
summary["correction helped?"] = summary["MAE as served"] < summary["MAE before correction"]
print("\nPer station, against complete days:")
display(summary)

learn = scored[scored["mode"].isin(bc.BIAS_CORRECT_MODES)]           # ▶ APP CODE: scripts/forecast/bias_correction.py · BIAS_CORRECT_MODES (what it learns from)
print(f"\nScored station-days the correction learns from (labelled {'/'.join(bc.BIAS_CORRECT_MODES)}): "
      f"{len(learn):,} of {len(scored):,}")
display(scored.assign(learned_from=scored["mode"].isin(bc.BIAS_CORRECT_MODES))
              .groupby(["city", "pollutant", "learned_from"]).size().unstack(fill_value=0))

errors = bc.city_day_errors(fc_hist, actuals, city_of)                 # ▶ APP CODE: scripts/forecast/bias_correction.py · city_day_errors (forecast/outlook rows only)
now = bc.decide_all({k: v for k, v in errors.items() if k[0] in CITIES},   # ▶ APP CODE: scripts/forecast/bias_correction.py · decide_all (decide, plus a reasoned skip for cities with nothing to learn from)
                    [(c, p) for c in CITIES for p in POLLUTANTS], TODAY, nf.BIAS_HISTORY_DAYS)
print("What today's bias_correction.decide_all concludes from the same history:")
pd.DataFrame([{"city": c, "pollutant": p, **d} for (c, p), d in sorted(now.items())])

**How to read this.** "MAE as served" below "MAE before correction" means the correction helped. A very large stored offset for PM2.5, especially a negative one pushing forecasts **up**, is the signature of the pre-audit bug that scored PM2.5 forecasts against AQI. If today's `decide` gives a much smaller offset, or does not apply one, the fix is doing its job and the stored decision will change once the new nightly runs. The stored decisions before Nov 2026 also learned from *seasonal_normal* rows; a city whose rows were mostly *seasonal_normal* can therefore flip between the stored and today's decision.

## 6. The diurnal (hourly) curve: does it earn its place?

The app turns each daily number into 24 hours by multiplying by a per-city ratio for that month and IST hour (`diurnal_shape`, applied by `expandDay`). This tests that step on its own with `backtest_hour`. A named hour on the next day is forecast as (a) **flat**, the day's number at every hour, or (b) **shape**, the day's number × the ratio. The day's number is held fixed (persistence of the daily mean), so only the hourly step is being judged. It is scored over the held-out season (city-mean hourly XKDR data; the shape is fitted on earlier years).

> **Limitations.** Hourly data are **city means**. The app applies the city's shape to every station, and there is no station-level hourly history locally to test that. `backtest_hour` is written for PM2.5. For AQI the notebook passes the AQI column under the PM2.5 name, so the same function scores it.

In [ ]:
hourly = {"pm25": B.load_city_hourly(),          # ▶ APP CODE: scripts/forecast/baselines.py · load_city_hourly (local cache)
          "aqi": B.load_city_hourly_aqi()}       # ▶ APP CODE: scripts/forecast/baselines.py · load_city_hourly_aqi (local cache)

def hourly_for(city, pol):
    g = hourly[pol][hourly[pol].city == city].copy()
    if pol == "aqi":   # same function, AQI in the column it reads; the AQI cache has no timestamp column, so build one
        g = g.rename(columns={"aqi": "pm25"}).assign(ts=lambda x: x.d + pd.to_timedelta(x.hr, unit="h"))
    return g

rows, per_hour = [], {}
for city in CITIES:
    for pol in POLLUTANTS:
        bt = B.backtest_hour(hourly_for(city, pol), REPLAY_YEAR, hours=range(24))   # ▶ APP CODE: scripts/forecast/baselines.py · backtest_hour
        if bt.empty:
            continue
        p = bt.pivot(index="hour", columns="model", values="mae")
        per_hour[(city, pol)] = p
        rows.append({"city": city, "pollutant": pol, "MAE flat": p.flat.mean(), "MAE with shape": p["shape"].mean(),
                     "gain from shape %": 100 * (1 - p["shape"].mean() / p.flat.mean()),
                     "hours where shape wins": int((p["shape"] < p.flat).sum())})
display(pd.DataFrame(rows).round(1))

fig, axes = plt.subplots(1, len(per_hour), figsize=(5 * len(per_hour), 3), squeeze=False)
for ax, ((city, pol), p) in zip(axes[0], per_hour.items()):
    ax.plot(p.index, p.flat, color=COLORS["climatology"], label="flat day")
    ax.plot(p.index, p["shape"], color=COLORS["blend"], label="day × diurnal shape")
    ax.set_title(f"{city} {pol}: next-day error by IST hour", fontsize=10); ax.set_xlabel("IST hour"); ax.set_ylabel(f"MAE ({UNITS[pol]})")
axes[0, 0].legend(fontsize=8); plt.tight_layout(); plt.show()

### Is the shape the app serves the shape the refit fits?

The monthly refit on CI (`--source db`) cannot refit the shape, because `readings_daily` has no hours, so it leaves the stored `diurnal_shape` rows alone. This compares those stored rows with what `fit_city` fits from the local hourly data (all years, as the parquet refit does).

In [ ]:
served_shape = vh.fetch_all(ro, "diurnal_shape", "city, pollutant, month, hour, ratio, n_days",
                            order=("city", "pollutant", "month", "hour"))
fig, axes = plt.subplots(1, len(CITIES) * len(POLLUTANTS), figsize=(5 * len(CITIES) * len(POLLUTANTS), 3), squeeze=False)
k = 0
for city in CITIES:
    for pol in POLLUTANTS:
        col = "aqi" if pol == "aqi" else "pm25"
        g = B.complete_days(hourly[pol][hourly[pol].city == city], value_col=col)   # ▶ APP CODE: scripts/forecast/baselines.py · complete_days (hourly, ≥20 h)
        refit = B.diurnal_shape(g, value_col=col)                                   # ▶ APP CODE: scripts/forecast/baselines.py · diurnal_shape (as fit_city calls it)
        refit = refit.rename_axis(["month", "hour"])                               # same (month, hour) index names as the stored table
        live = served_shape[(served_shape.city == city) & (served_shape.pollutant == pol)].set_index(["month", "hour"]).ratio
        both = pd.concat({"served": live, "refit": refit}, axis=1).dropna()
        diff = (both.served - both.refit).abs()
        print(f"{city} {pol}: {len(live)} served ratios; largest difference from a fresh fit {diff.max():.3f}, mean {diff.mean():.4f}")
        ax = axes[0, k]; k += 1
        nov = both.xs(11, level="month")
        ax.plot(nov.index, nov.served, "o-", ms=3, label="served (Supabase)"); ax.plot(nov.index, nov.refit, "--", label="refit from parquet")
        ax.axhline(1, color="#aaa", lw=0.8); ax.set_title(f"{city} {pol}: November shape", fontsize=10); ax.set_xlabel("IST hour")
axes[0, 0].legend(fontsize=8); plt.tight_layout(); plt.show()

## 7. Averaging over the three nearest stations

A user asks about a **place**. The app finds the 3 nearest live stations within 25 km (`nearest_monitors`) and takes the plain mean of their forecasts for each date. Stations serving the seasonal normal are left out whenever a neighbour has a real forecast (`averageForecastDays`). Is the average better than just using the single nearest station? "averaged" and "nearest only" use the bias-corrected numbers from §4 (what the app shows); "averaged (before bias corr.)" is the same average of the uncorrected model output.

Truth at the user's point is unknown, because there is no monitor there. Two stand-ins are used: the **nearest station's** reading (the closest thing to the user's air) and the **mean of the three** stations' readings.

> **Limitations.** (1) Only `LOCATIONS_PER_CITY` random points per city (fixed seed): each is a random station's position moved up to 5 km in a random direction. The app accepts any point. (2) The search and the averaging are the Python mirror (`serving_mirror.py`), pinned to the TypeScript by the shared test. Its distance uses a sphere, where PostGIS uses the ellipsoid. (3) "Live" here means *the station has data in the held-out season*. The app's test (an OpenAQ day in the last 14 days) only makes sense for today. (4) Only stations of the same city are searched, because only those were replayed in §4. The app does not filter by city.

In [ ]:
def random_points(city, n, rng):
    # A random station's position, moved 0–5 km in a random direction.
    st = [s for s in vh.manifest_stations() if s["analysis_city"] == city]
    out = []
    for _ in range(n):
        s = st[rng.integers(len(st))]
        km, bearing = rng.uniform(0, 5), rng.uniform(0, 2 * np.pi)
        lat = s["latitude"] + (km / 111.2) * np.cos(bearing)
        lng = s["longitude"] + (km / (111.2 * np.cos(np.radians(s["latitude"])))) * np.sin(bearing)
        out.append((round(lat, 4), round(lng, 4)))
    return out, st

rng = np.random.default_rng(RANDOM_SEED)
rows, examples = [], {}
for (city, pol), rep in replays.items():
    r1 = rep[rep.horizon == 1]
    points, city_stations = random_points(city, LOCATIONS_PER_CITY, rng)
    live = set(r1.station)                                    # stand-in for "has recent data"
    by_station = {s: g.set_index("target") for s, g in r1.groupby("station")}
    for lat, lng in points:
        near = mirror.nearest_stations(lat, lng, city_stations, k=3, max_km=25, live_ids=live)   # ▶ APP CODE (mirror of SQL nearest_monitors): serving_mirror.nearest_stations
        if not near:
            continue
        rec = []
        for t in sorted(r1.target.unique()):
            night = (t - pd.Timedelta(days=1)).date().isoformat()
            per, per_bc = [], []
            for m in near:
                g = by_station.get(m["id"])
                if g is None or t not in g.index:
                    continue
                x = g.loc[t]
                row = {"pollutant": pol, "target_date": t.date().isoformat(), "horizon_days": 1, "value": x.served,
                       "band_p80": None, "mode": x["mode"], "model": x.model,
                       "based_on_date": None, "data_age_days": x.data_age}
                per.append({"monitor": m, "days": [mirror.expand_day(row, city, {}, night, hourly=False)]})  # ▶ APP CODE (mirror of lib/api/forecast.ts): expandDay
                # The same row after the bias correction: what the app would really have averaged.
                per_bc.append({"monitor": m, "days": [mirror.expand_day({**row, "value": x.served_bc}, city, {}, night, hourly=False)]})  # ▶ APP CODE (mirror of lib/api/forecast.ts): expandDay
            avg = mirror.average_forecast_days(per, 1)                                                     # ▶ APP CODE (mirror of lib/api/data.ts): averageForecastDays
            avg_bc = mirror.average_forecast_days(per_bc, 1)                                               # ▶ APP CODE (mirror of lib/api/data.ts): averageForecastDays
            g0 = by_station.get(near[0]["id"])
            if avg is None or g0 is None or t not in g0.index or pd.isna(g0.loc[t].actual):
                continue
            actual3 = np.nanmean([by_station[m["id"]].loc[t].actual for m in near
                                  if m["id"] in by_station and t in by_station[m["id"]].index])
            rec.append({"target": t, "averaged": avg_bc["value"][0]["value"],
                        "averaged (before bias corr.)": avg["value"][0]["value"],
                        "nearest only": g0.loc[t].served_bc,
                        "seasonal normal (nearest)": g0.loc[t].climatology,
                        "actual (nearest)": g0.loc[t].actual, "actual (mean of 3)": actual3})
        d = pd.DataFrame(rec)
        if d.empty:
            continue
        examples[(city, pol, lat, lng)] = d
        out = {"city": city, "pollutant": pol, "point": f"{lat}, {lng}",
               "stations (km)": ", ".join(f"{vh.manifest_name(m['id'])[:18]} ({m['distance_km']})" for m in near),
               "days": len(d)}
        for truth in ("actual (nearest)", "actual (mean of 3)"):
            for what in ("averaged", "averaged (before bias corr.)", "nearest only", "seasonal normal (nearest)"):
                out[f"MAE {what} vs {truth.split('(')[1][:-1]}"] = (d[what] - d[truth]).abs().mean()
        rows.append(out)
pd.DataFrame(rows).round(1)

**How to read this.** Compare "averaged" with "nearest only" within each truth column. Averaging should win clearly against the mean-of-3 truth, which is partly by construction. Against the nearest station's own reading it may win or lose: it trades some local detail for less noise. Both should beat the seasonal normal. A point that gets only one station (the others too far away or not live) behaves exactly like "nearest only".

**The Ask AI answers use this same averaging.** Since Nov 2026 (commit 28bfc5c, and its follow-up for `best_hour` and `rest_of_today`), the AI's `forecast`, `best_hour` and `rest_of_today` tools call the app's own `averageForecastDays` (`lib/api/data.ts`) instead of averaging stations themselves. So an AI answer about a place equals what the forecast screen shows for it, and is covered by the mirror and the shared test above; it needs no separate Python mirror. If an AI tool ever averages stations on its own again, it would need one.

In [ ]:
if examples:
    (city, pol, lat, lng), d = next(iter(examples.items()))
    d = d.set_index("target").loc[f"{REPLAY_YEAR}-10-01":]
    plt.figure(figsize=(12, 3))
    plt.plot(d.index, d["actual (nearest)"], color=COLORS["actual"], label="actual at nearest station")
    plt.plot(d.index, d["averaged"], color=COLORS["blend"], label="served: average of nearest 3")
    plt.plot(d.index, d["nearest only"], color=COLORS["persistence"], lw=0.9, label="nearest station only")
    plt.plot(d.index, d["seasonal normal (nearest)"], color=COLORS["climatology"], ls="--", label="seasonal normal")
    plt.title(f"{city} {pol} at ({lat}, {lng}): +1 day, reading {DATA_AGE_FOR_PLOTS} d old", fontsize=10)
    plt.ylabel(UNITS[pol]); plt.legend(fontsize=8, ncol=4); plt.tight_layout(); plt.show()

## 8. Live match: does last night's stored forecast equal what the code computes?

This is the check that the quality measured above is the quality actually **served**. For a sample of stations it reloads the stored parameters and mode table and the `readings_daily` rows the nightly job would have seen, and recomputes the latest night's forecast with `build_station_rows`. It then compares the result with `forecast_daily`, using `value_raw` (before bias correction) where it exists and `value` otherwise.

> **Expected mismatch, for now.** Until Amrita runs the post-audit refit **and** nightly on `main`, `forecast_daily` was written by the **old** code: α capped at the 3-day value, the blend always, no completeness rule. The cell detects this from the stored tables (§3: empty `forecast_params.alphas` / `forecast_modes.model`) and reports it plainly instead of flagging a failure. Even with the new code, a row can differ legitimately when data for an older day arrived **after** the run, which changes the starting reading. Those rows are counted separately.

In [ ]:
upcoming = vh.fetch_all(ro, "forecast_daily",
                        "monitor_id, pollutant, target_date, horizon_days, value, value_raw, mode, model, based_on_date, data_age_days, computed_at",
                        filters=[("gt", "target_date", (TODAY - timedelta(days=1)).isoformat()),
                                 ("in_", "pollutant", POLLUTANTS)],
                        order=("monitor_id", "pollutant", "target_date"))
upcoming["run_date"] = (pd.to_datetime(upcoming.target_date) - pd.to_timedelta(upcoming.horizon_days, unit="D")).dt.date
run_date = upcoming.run_date.mode().iat[0]
last_run = upcoming[upcoming.run_date == run_date]
print(f"Latest nightly run: {run_date} (computed at {last_run.computed_at.max()}); {len(last_run):,} rows for {POLLUTANTS}")

# The readings_daily rows that run could have seen: the 14 days before it, nothing from the run day onward.
rows = nf.fetch_daily_rows(ro, run_date - timedelta(days=nf.LOOKBACK_DAYS))     # ▶ APP CODE: scripts/forecast/nightly_forecast.py · fetch_daily_rows
daily = pd.DataFrame(rows).assign(date=lambda d: pd.to_datetime(d["date"]))
observations = sim.latest_observation(daily, run_date, tuple(POLLUTANTS))       # ▶ APP CODE: scripts/forecast/simulate.py · latest_observation (nightly's observed_values, strictly before the run day)

rng = np.random.default_rng(RANDOM_SEED)
sample = []
for city in CITIES:
    ids = sorted({s["monitor_id"] for s in vh.manifest_stations() if s["analysis_city"] == city})
    sample += [(m, city) for m in rng.choice(ids, size=min(SAMPLE_STATIONS_PER_CITY, len(ids)), replace=False)]

cmp = []
for monitor_id, city in sample:
    for pol in POLLUTANTS:
        p = params_live.get((monitor_id, pol))
        if p is None:
            continue
        mine = nf.build_station_rows(monitor_id, city, pol, p, observations.get((monitor_id, pol)),   # ▶ APP CODE: scripts/forecast/nightly_forecast.py · build_station_rows
                                     modes_live, run_date, nf.DEFAULT_HORIZON_DAYS)
        stored = last_run[(last_run.monitor_id == monitor_id) & (last_run.pollutant == pol)].set_index("target_date")
        for r in mine:
            if r["target_date"] not in stored.index:
                continue
            s = stored.loc[r["target_date"]]
            stored_value = s.value_raw if pd.notna(s.value_raw) else s.value
            cmp.append({"station": vh.manifest_name(monitor_id)[:22], "pollutant": pol, "h": r["horizon_days"],
                        "recomputed": r["value"], "stored": stored_value,
                        "recomputed mode/model": f"{r['mode']}/{r['model']}", "stored mode/model": f"{s['mode']}/{s.model}",
                        "recomputed based on": r["based_on_date"], "stored based on": s.based_on_date})
cmp = pd.DataFrame(cmp)
cmp["match"] = (cmp.recomputed - cmp.stored).abs() <= 0.01
cmp["different reading"] = cmp["recomputed based on"].fillna("none") != cmp["stored based on"].fillna("none")

n, n_ok = len(cmp), int(cmp.match.sum())
n_reading = int((~cmp.match & cmp["different reading"]).sum())
print(f"\n{n_ok}/{n} recomputed rows match the stored forecast; {n_reading} of the mismatches start from a different reading.")
if not status["new_code"]:
    print("EXPECTED MISMATCH — not a failure. forecast_params / forecast_modes were written by the pre-audit refit, "
          "so forecast_daily was made by the pre-audit nightly. Re-run this cell after the new refit and nightly "
          "have run on main; then every row should match apart from late-arriving data.")
elif n_ok + n_reading == n:
    print("PASS: every row matches, apart from rows whose starting reading changed because data arrived after the run.")
else:
    print(f"FAIL: {n - n_ok - n_reading} rows differ with the same starting reading. The served forecast is not "
          "what the code computes. Investigate before trusting §2–§4.")
print("Share of rows that match, by horizon:", cmp.groupby("h").match.mean().round(2).to_dict())
cmp[~cmp.match].head(20)

**How to read this.** With the pre-audit tables, rows whose *effective horizon* (data age + days ahead) is 3 or less should still match. The old and new code use the same α there. Rows beyond that differ: the old code reused the 3-day α, while the new code has no α fitted for them in the old tables and serves the seasonal normal. The stored *model* label can also differ. The old nightly copied `forecast_params.model` into the label but always computed the blend (audit D5).

## 9. Recent track record: what was served, and what stations then measured

The last `RECENT_NIGHTS` nights of served +1-day forecasts, scored against complete station-days. The seasonal normal (each station's stored climatology for that date) is scored alongside, so the headline is again **skill vs the seasonal average**. The ages of the readings the forecasts started from are shown too: they decide how much skill §3 says is available.

> **Limitation.** These rows were written by whatever code was live on each night (see §8). The seasonal normal uses today's stored climatology, which may have been refitted since a given night.

In [ ]:
start = TODAY - timedelta(days=RECENT_NIGHTS)
served = vh.fetch_all(ro, "forecast_daily",
                      "monitor_id, pollutant, target_date, value, value_raw, mode, model, data_age_days",
                      filters=[("eq", "horizon_days", 1), ("in_", "pollutant", POLLUTANTS),
                               ("gte", "target_date", start.isoformat()), ("lt", "target_date", TODAY.isoformat())],
                      order=("target_date", "monitor_id", "pollutant"))
served["city"] = served.monitor_id.map(city_of)
served = served[served.city.isin(CITIES)].copy()
served["actual"] = [actuals.get((m, d, p)) for m, d, p in zip(served.monitor_id, served.target_date, served.pollutant)]
served["seasonal normal"] = [params_live[(m, p)]["climatology"][pd.Timestamp(d).dayofyear] if (m, p) in params_live else np.nan
                             for m, d, p in zip(served.monitor_id, served.target_date, served.pollutant)]
scored9 = served.dropna(subset=["actual"])

def summarise(g):
    mae = (g.value - g.actual).abs().mean()
    clim = (g["seasonal normal"] - g.actual).abs().mean()
    return pd.Series({"station-days": len(g), "MAE served": mae,
                      "MAE before bias corr.": (g.value_raw.fillna(g.value) - g.actual).abs().mean(),
                      "MAE seasonal normal": clim, "skill vs seasonal %": 100 * (clim - mae) / clim,
                      "median data age (d)": g.data_age_days.median()})

print(f"Served +1-day forecasts {start} → {TODAY - timedelta(days=1)} with a complete actual day: {len(scored9):,} of {len(served):,}")
display(scored9.groupby(["city", "pollutant"]).apply(summarise).round(1))
print("By the label the app showed:")
excluded = scored9[~scored9["mode"].isin(bc.BIAS_CORRECT_MODES)]      # ▶ APP CODE: scripts/forecast/bias_correction.py · BIAS_CORRECT_MODES
n_shifted = int(excluded.value_raw.notna().sum())
print(f"  rows labelled outside {bc.BIAS_CORRECT_MODES} that still carry a bias correction: {n_shifted} of {len(excluded):,}"
      + ("  (expected from nights before the forecast/outlook-only rule went live)" if n_shifted else ""))
display(scored9.groupby(["city", "pollutant", "mode"]).apply(summarise).round(1))

for (city, pol), g in scored9.groupby(["city", "pollutant"]):
    d = g.assign(raw=g.value_raw.fillna(g.value)).groupby("target_date")[["value", "raw", "actual", "seasonal normal"]].mean()
    d.index = pd.to_datetime(d.index)
    d = d.asfreq("D")
    plt.figure(figsize=(12, 2.8))
    plt.plot(d.index, d.actual, "o-", color=COLORS["actual"], label="measured (complete days)")
    plt.plot(d.index, d.value, "o-", color=COLORS["served"], label="served +1 day")
    plt.plot(d.index, d.raw, "o:", ms=3, color=COLORS["served"], alpha=0.6, label="before bias correction")
    plt.plot(d.index, d["seasonal normal"], "--", color=COLORS["climatology"], label="seasonal normal")
    plt.title(f"{city} {pol}: station average, last {RECENT_NIGHTS} nights", fontsize=10); plt.ylabel(UNITS[pol])
    plt.legend(fontsize=8, ncol=4); plt.tight_layout(); plt.show()
print(f"Notebook finished in {time.time() - T0:.0f} s")

**How to read this.** The headline is *skill vs seasonal %*: above 0 the served forecast beat the seasonal average over these nights. Compare "MAE served" with "MAE before bias corr.": if the served error is clearly larger, the bias correction made things worse over this period (§5 shows why). The correction only touches rows labelled *forecast* or *outlook*; rows labelled *seasonal_normal* are left as the seasonal average, so in the "by label" table their "MAE served" and "MAE before bias corr." should be equal. Nights written before that rule went live (mid-Nov 2026) corrected every row, including *seasonal_normal*; the printed count above shows how many such rows are in this window.

## Keeping this notebook current

This notebook is only useful while it calls the code the app runs. **Whenever forecasting logic changes, update this notebook in the same commit and re-run it.** That covers `scripts/forecast/*`, `lib/api/forecast.ts`, `averageForecastDays` in `lib/api/data.ts`, the `nearest_monitors` SQL, and `lib/daily-quality-config.json`.

- If the TypeScript serving layer changes, update `scripts/forecast/serving_mirror.py`, then regenerate `tests/fixtures/serving_mirror/cases.json` and check that **both** `tests/test_serving_mirror.py` and `tests/api/serving-mirror.test.ts` pass.
- If a function marked `▶ APP CODE` is renamed or its signature changes, fix the call here rather than copying the old logic in.
- After each monthly refit, run §3 and §8. §3 should agree with `forecast_modes`, and §8 should report PASS once the post-audit code is live.
- The held-out years come from `baselines.eligible_test_years` (the `SKILL_*` constants in `baselines.py`). Change the rule there, not here.
- Run `QUICK = True` for a fast look; run `QUICK = False` before relying on the numbers (roughly 8–15 minutes).